In [1]:
from Crypto.PublicKey import RSA
from Crypto.Random import get_random_bytes
from Crypto.Cipher import AES, PKCS1_OAEP
import socket
import select
import sys


def send_username():
    user = input("Username: ")
    data = user.encode('utf-8')

    cipher_aes = AES.new(session_key, AES.MODE_EAX)
    ciphertext, tag = cipher_aes.encrypt_and_digest(data)

    client_socket.sendall(cipher_aes.nonce)
    client_socket.sendall(tag)
    client_socket.sendall(ciphertext)
    print('Sent!')
    return user

def send_message():
    data = input(f'{user} > ')
    data = data.encode('utf-8')

    cipher_aes = AES.new(session_key, AES.MODE_EAX)
    ciphertext, tag = cipher_aes.encrypt_and_digest(data)

    client_socket.sendall(cipher_aes.nonce)
    client_socket.sendall(tag)
    client_socket.sendall(ciphertext)
    print('Sent!')

def receive_message():
    nonce = client_socket.recv(16)
    tag = client_socket.recv(16)
    ciphertext = client_socket.recv(1000)

    cipher_aes = AES.new(session_key, AES.MODE_EAX, nonce)
    data = cipher_aes.decrypt_and_verify(ciphertext, tag)
    return data

IP = "127.0.0.1"
PORT = 1232

recipient_key = RSA.import_key(open("receiver.pem").read())
session_key = get_random_bytes(16)

cipher_rsa = PKCS1_OAEP.new(recipient_key)
enc_session_key = cipher_rsa.encrypt(session_key)

client_socket = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
client_socket.connect((IP, PORT))
client_socket.setblocking(False)
client_socket.sendall(enc_session_key)

user = send_username()

while True:
    send_message()
    try:
        message = receive_message()
        print(message)
            
    except:
        print('error ups')
        continue

Username: ziga1
Sent!
ziga1 > hallo
Sent!
error ups
ziga1 > hello
Sent!
b'ziga2: hello'


KeyboardInterrupt: Interrupted by user